# Hotel Bar Inventory Forecasting & Par Level Recommendation System

This notebook runs the end-to-end workflow: load → validate → aggregate → EDA → forecast → par level → inventory simulation.


In [ ]:
from pathlib import Path
import sys
import os

import pandas as pd
import numpy as np

# Make the project root the working directory so src/ and data/ resolve correctly.
project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import Config
from src.data_loader import load_inventory_data
from src.preprocessing import validate_conservation, prepare_daily_consumption
from src.eda import abc_categorization, identify_stockout_days
from src.forecasting import compare_models, forecast_next_days
from src.par_level import compute_par_level
from src.inventory_simulation import simulate_bar_inventory

print("Project root:", project_root)
print("Project modules loaded successfully!")


In [ ]:
cfg = Config()
raw = load_inventory_data(cfg.raw_data_path)
raw.head()


## 1. Data validation

The inventory conservation equation is checked row by row.


In [ ]:
validated = validate_conservation(raw)
validated[['Opening Balance','Purchase','Consumed','Closing Balance','conservation_difference','conservation_ok']].head()


In [ ]:
print('Rows:', len(validated))
print('Conservation pass rate:', round(validated['conservation_ok'].mean()*100, 2), '%')


## 2. Daily aggregation and complete date grid


In [ ]:
daily = prepare_daily_consumption(raw)
daily.head(10)


## 3. ABC categorization and stockout audit


In [ ]:
abc = abc_categorization(daily)
abc.head(15)


In [ ]:
stockouts = identify_stockout_days(validated)
print('Historical stockout rows:', len(stockouts))


## 4. Forecast model comparison

The split is chronological: the earlier observations are training data and the final observations are validation data.


In [ ]:
first_key = next(iter(daily.groupby(['Bar Name','Brand Name']).groups))
bar, brand = first_key
series = (daily[(daily['Bar Name']==bar) & (daily['Brand Name']==brand)]
          .sort_values('Date')['Consumed (ml)'].reset_index(drop=True))
print('Series:', bar, '/', brand, 'days:', len(series))
results, train, test = compare_models(series)
results


## 5. Dynamic par level


In [ ]:
forecast = forecast_next_days(series, horizon=cfg.lead_time_days, model_name='Holt-Winters')
std_daily = float(series.tail(28).std())
std_daily = 0.0 if np.isnan(std_daily) else std_daily
par = compute_par_level(
    predicted_daily_demand=float(np.mean(forecast)),
    std_daily_demand=std_daily,
    lead_time_days=cfg.lead_time_days,
    service_level_z=cfg.service_level_z,
)
par


## 6. Historical inventory backtest


In [ ]:
simulation = simulate_bar_inventory(
    actual_demand=series.values,
    par_level=par['par_level'],
    lead_time=cfg.lead_time_days,
    initial_stock=par['par_level'],
)
pd.Series({k:v for k,v in simulation.items() if k != 'history'})


## 7. Operational interpretation

The output gives a forecast-informed replenishment target. Run the full pipeline with `python run.py` to generate the project artifacts under `data/processed/` and `reports/`.
